## RAG pipeline - Data Ingestion to Vector DB

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/var/folders/kv/f0xmp3wd0n51ydqcckq6wpyh0000gn/T/ipykernel_8870/3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
/Users/varun_asija/Documents/AI/rag-pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data Ingestion

In [2]:
## Read all pdfs inside the directory
def process_all_pdfs(dir:str):
    """"Process all the pdfs inside the directory and return the list of documents"""
    all_documents=[]
    pdf_dir = Path(dir)

    #Find all pdfs recursively inside the directory
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} pdf files in the directory {dir}")

    for pdf_file in pdf_files:
        print(f"Processing: {pdf_file.name}")
        loader = PyPDFLoader(pdf_file)
        documents = loader.load() # N documents for N pages in the pdf
        print(f"+ Loaded: {len(documents)}")
        
        # Add source metadata to each document
        for doc in documents:
            doc.metadata["source_file"] = pdf_file.name
            doc.metadata["file_type"]   = "pdf"

        all_documents.extend(documents)
    print(f"Processed {len(all_documents)} documents from {len(pdf_files)} pdf files")
    return all_documents

In [3]:
all_pdf_documents = process_all_pdfs("../data")

Found 4 pdf files in the directory ../data
Processing: rag_handbook.pdf
+ Loaded: 5
Processing: aws_data_lake.pdf
+ Loaded: 5
Processing: python_reference.pdf
+ Loaded: 5
Processing: ml_guide.pdf
+ Loaded: 5
Processed 20 documents from 4 pdf files


## Data Chunking

In [4]:
def chunk_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into chunks of specified size with overlap"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )

    chunks = text_splitter.split_documents(documents)
    print(f"Splited {len(documents)} documents into {len(chunks)} chunks")

    # Show example of a chunk
    if chunks:
        print("\nExample chunk:")
        print(f"Content: {chunks[0].page_content[:200]}...")
        print(f"Metadata: {chunks[0].metadata}")
    return chunks


In [5]:
chunks = chunk_documents(all_pdf_documents, chunk_size=1000, chunk_overlap=200)

Splited 20 documents into 91 chunks

Example chunk:
Content: Rag Handbook
Introduction
RAG combines retrieval and generation. It indexes documents, embeds chunks, retrieves relevant
passages, and augments prompts. Paragraph 1. Engineers document architecture, a...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-06T13:54:00+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-07-06T13:54:00+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/rag_handbook.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'rag_handbook.pdf', 'file_type': 'pdf'}


## Embedding and storing into vector store database

In [6]:
import os
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingGenerator:
    "Handle document embedding generation using Sentence Transformers"
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding generator
        Args:
            model_name (str): Hugging Face model name for sentence transformers. Default is all-MiniLM-L6-v2
            """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentence transformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension is: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error occurred while loading the model: {e}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts
        Args:
            texts (List[str]): List of text strings to generate embeddings for"""
        if not self.model:
            raise ValueError("Model is not loaded.")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

In [8]:
embedding_manager = EmbeddingGenerator()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 19746.47it/s]


Model loaded successfully. Embedding dimension is: 384


/var/folders/kv/f0xmp3wd0n51ydqcckq6wpyh0000gn/T/ipykernel_8870/2235720056.py:17: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension is: {self.model.get_sentence_embedding_dimension()}")


#### Vector Store using ChromaDB

In [9]:
class VectorStore:
    """Handle vector store operations using ChromaDB"""
    def __init__(self, collection_name: str= "pdf_documments", persist_directory: str = "../data/chroma_db"):
        """Initilize the vector store
        Args:
            collection_name (str): Name of the collection in ChromaDB. Default is "pdf_documments"
            persist_directory (str): Directory to persist the ChromaDB database. Default is "../data/chroma_db"
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the ChromaDB client and collection"""
        try:
            # create perisist chromadb client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # create collection if not exists
            self.collection = self.client.get_or_create_collection(name=self.collection_name, 
                                                                   metadata={"description": "PDF documents embedding for RAG",
                                                                             "hnsw:space": "cosine"})
        
            print(f"Vector store initialized with collection: {self.collection_name}")
            print(f"Existing collection size: {self.collection.count()}")
        except Exception as e:
            ValueError(f"Error occurred while initializing the vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and their embeddings to the vector store
        Args:
            documents (List[Any]): List of langchain documents
            embeddings (np.ndarray): Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents must match the number of embeddings.")

        print(f"Adding {len(documents)} documents to the vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate a unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # document text
            documents_texts.append(doc.page_content)

            # embedding
            embeddings_list.append(embedding.tolist())

        try:
            # add to collection
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_texts,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total collection size after addition: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error occurred while adding documents to the vector store: {e}")
            raise

In [10]:
vector_store = VectorStore()

Vector store initialized with collection: pdf_documments
Existing collection size: 182


In [11]:
# convert text to embedding
# First we need to extract the text from the documents
texts = [doc.page_content for doc in chunks]

# Generate embeddings for the extracted texts
embeddings = embedding_manager.generate_embeddings(texts)

# store the documents and embeddings in the vector store
vector_store.add_documents(chunks, embeddings)

Batches: 100%|██████████| 3/3 [00:00<00:00,  7.67it/s]

Generated embeddings with shape: (91, 384)
Adding 91 documents to the vector store...
Successfully added 91 documents to the vector store.
Total collection size after addition: 273


## Retrieval Pipeline from VectorStore

In [12]:
from typing import List, Dict, Any
class RagRetriever:
    """Handle query-based retrieval from vector store."""
    def __init__(self, vector_store, embedding_manager):
        """Initialize the RagRetriever with a vector store and an embedding manager.
        
        Args:
            vector_store: An instance of the vector store to retrieve documents from.
            embedding_manager: An instance of the embedding manager to generate embeddings.
        """
        self.vector_store = vector_store
        self.emebedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, similarity_threshold: float =0.0) -> List[Dict[str, Any]]:
        """Retrieve relevant documents based on the query.
        
        Args:
            query: The input query string for which to retrieve documents.
            top_k: The number of top documents to retrieve (default is 5).
            similarity_threshold: The minimum similarity score for a document to be considered relevant (default is 0.0).
        
        Returns:
            A list of retrieved documents that meet the similarity threshold.
        """
        # Generate embedding for the query
        query_embedding = self.emebedding_manager.generate_embeddings([query])[0]

        # Retreive documents from the vector store based on the query embedding
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []
            # Filter results based on the similarity threshold
            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadata = results["metadatas"][0]
                distance = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadata, distance)):
                    # convert distance to similarity score
                    similarity_score = 1 - distance
                    
                    if similarity_score >= similarity_threshold:
                        retrieved_docs.append(
                            {
                                "id": doc_id,
                                "content": document,
                                "metadata": metadata,
                                "similarity_score": similarity_score,
                                "distance": distance,
                                "rank": i + 1
                            }
                        )
                print(f"retrieved docs: {len(retrieved_docs)}")
            else:
                print("No documents retrieved for the given query.")
            return retrieved_docs
        except Exception as e:
            print(f"Error occurred during retrieval: {e}")
            return []

In [13]:
rag_retriever = RagRetriever(vector_store, embedding_manager)

In [14]:
# rag_retriever.retrieve("What is rag")

## Integrate VectorDB context with LLM output

In [15]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

True

In [27]:
# Initialize the ChatGroq model with the API key from environment variables
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.1, max_tokens=1024)

# Simple rag function: retrieve context + generate response
def rag_pipeline(query, retriver, llm, top_k = 3):
    ## retrieve the context
    results = rag_retriever.retrieve(query, top_k=top_k)

    context = "\n\n".join([doc["content"] for doc in results]) if results else ""
    sources = [{"source": doc["metadata"].get("source_file"),
                "page":  doc["metadata"].get("page")
                }
                for doc in results]
    if not context:
        return "No relevant context found to answer the question"
    ## generate the answer using GROQ LLM
    prompt = f"""
    You are a question-answering assistant.

    Answer ONLY using the information provided in the context below.

    If the answer cannot be found in the context, reply exactly:

    "I don't have enough information in the provided context."

    Do not use your own knowledge.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """
    response=llm.invoke([prompt.format(context=context, query=query)])
    output = {"answer": response.content,
              "sources": sources}
    return output


In [30]:
rag_pipeline("what is datalake", retriver=rag_retriever, llm=llm)

Batches: 100%|██████████| 1/1 [00:00<00:00, 28.94it/s]


Generated embeddings with shape: (1, 384)
retrieved docs: 3


{'answer': 'Data lakes separate bronze, silver and gold layers.',
 'sources': [{'source': 'aws_data_lake.pdf', 'page': 0},
  {'source': 'aws_data_lake.pdf', 'page': 0},
  {'source': 'aws_data_lake.pdf', 'page': 0}]}